# Train ICARUS CAE — GPU slice B

**Kernel:** `conda env:.conda-diffusion` (one GPU / MIG slice)

Companion: `04a_TrainVAE_ICARUS.ipynb` on the other slice.

Data: `/exp/sbnd/data/users/gputnam/DNN-ROI-images/`  
Flags: `configs/train_flags/cae_icarus.sh`  
Logs: scratch `.../training/cae/icarus/` (synced to `DATA_ROOT/training/cae/icarus/`)

**Before launch:** stop leftover 01a/01b trainers and clear the stop flag (on EAF):
```bash
bash train/stop_running_trainers.sh
rm -f /exp/sbnd/app/users/munjung/anomaly-detection/STOP_TRAINING
```

### Modes (set `MODE` in the next cell)
| `MODE` | What happens |
|--------|----------------|
| `"dry"` | Write `launch.sh` only — **no training** (default) |
| `"smoke"` | Short GPU run (~5 steps) → tiny checkpoint |
| `"full"` | Train to `MAX_STEPS` (612000) |


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))
if "configs.paths" in sys.modules:
    importlib.reload(sys.modules["configs.paths"])

from configs.paths import (
    DATA_ROOT,
    GPUTNAM_DNN_ROI,
    describe_environment,
    ensure_layout,
    existing_scratch_pools,
    resolve_ae_training_run_root,
)

ensure_layout()
env = describe_environment()
print(json.dumps(env, indent=2))
print("scratch pools:", [str(p) for p in existing_scratch_pools()] or ["<none>"])
print("DNN-ROI:", GPUTNAM_DNN_ROI, "exists=", GPUTNAM_DNN_ROI.is_dir())
assert env.get("on_eaf") or existing_scratch_pools(), (
    "Expected EAF scratch. Open this notebook on jupyter-munjung with the diffusion kernel."
)

DIFFUSION_ROOT = APP_ROOT / "train" / "diffusion-anomaly"
FLAG_SCRIPT = APP_ROOT / "configs" / "train_flags" / "cae_icarus.sh"
AE_TYPE = "cae"
assert FLAG_SCRIPT.is_file(), FLAG_SCRIPT
assert "icarus" in FLAG_SCRIPT.name and "sbnd" not in FLAG_SCRIPT.name, FLAG_SCRIPT

# --- ICARUS samples (NOT SBND) ---
# Flag DATADIR → /exp/sbnd/data/users/gputnam/DNN-ROI-images/
# Format: event_*/deconvolved_signal (ICARUS). SBND raw would be event_*/raw.
DATA_DIR = GPUTNAM_DNN_ROI
assert DATA_DIR.is_dir(), DATA_DIR
h5s = sorted(DATA_DIR.glob("*.h5"))
assert h5s, f"No .h5 under {DATA_DIR}"
import h5py
with h5py.File(h5s[0], "r") as _f:
    _ev = next(iter(_f.keys()))
    assert "deconvolved_signal" in _f[_ev], (
        f"{h5s[0]} looks like SBND/raw, not ICARUS DNN-ROI (missing deconvolved_signal)"
    )
print(f"TRAINING DATA (ICARUS DNN-ROI): {DATA_DIR}")
print(f"  n_h5={len(h5s)}  example={h5s[0].name}  format=deconvolved_signal")

# --- controls ---
# MODE: "dry" | "smoke" | "full"
MODE = "full"          # ← launching full ICARUS CAE training
RESUME = True
MAX_STEPS = "612000"
BATCH_SIZE = None      # None → flag script (16); set 4/8 on tight MIG

DRY_RUN = MODE == "dry"
SMOKE_TEST = MODE == "smoke"
print(f"MODE={MODE}  DRY_RUN={DRY_RUN}  SMOKE_TEST={SMOKE_TEST}")


{
  "hostname": "jupyter-munjung-anomaly-detection",
  "on_eaf": true,
  "app_root": "/exp/sbnd/app/users/munjung/anomaly-detection",
  "data_root": "/exp/sbnd/data/users/munjung/anomaly-detection",
  "data_root_exists": true,
  "scratch_pools": [
    "/scratch/7DayLifetime"
  ],
  "scratch_root": "/scratch/7DayLifetime/munjung/anomaly-detection",
  "scratch_exists": true,
  "scratch_icarus": "/scratch/7DayLifetime/munjung/ICARUS",
  "scratch_icarus_exists": true,
  "diffusion_code_root": "/exp/sbnd/app/users/munjung/anomaly-detection/train/diffusion-anomaly",
  "default_checkpoint": "/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt",
  "default_checkpoint_exists": true,
  "vae_sbnd_checkpoint": "None",
  "cae_sbnd_checkpoint": "None",
  "vae_icarus_checkpoint": "None",
  "cae_icarus_checkpoint": "None",
  "gputnam_dnn_roi": "/exp/sbnd/data/users/gputnam/DNN-ROI-images",
  "gputnam_dnn_roi_exists": true
}
scratch pools: ['/scratch/7DayLifetime']
DNN-ROI: /

In [2]:
stop = APP_ROOT / "STOP_TRAINING"
if stop.is_file():
    stop.unlink()
    print(f"removed stop file {stop}")

LOG_DIR = resolve_ae_training_run_root(AE_TYPE, "icarus")
DURABLE = DATA_ROOT / "training" / AE_TYPE / "icarus"
DURABLE.mkdir(parents=True, exist_ok=True)
print("LOG_DIR ", LOG_DIR)
print("DURABLE ", DURABLE)
print("DATA_DIR (ICARUS)", DATA_DIR)


def latest_resume(log_dir: Path, durable: Path) -> Path | None:
    cands = []
    for d in (log_dir, durable):
        if not d.is_dir():
            continue
        cands += sorted(d.glob("brats2update*.pt"))
        cands += sorted(d.glob("model*.pt"))
    return cands[-1] if cands else None


resume_ckpt = latest_resume(LOG_DIR, DURABLE) if RESUME else None
print("resume:", resume_ckpt)

# Explicit ICARUS data paths last so they win over anything in the flag script.
extras = [
    f"--data_dir {DATA_DIR}",
    f"--validation_dir {DATA_DIR}",
    f"--max_steps {MAX_STEPS}",
]
if BATCH_SIZE is not None:
    mb = min(int(BATCH_SIZE), 4)
    extras += [f"--batch_size {BATCH_SIZE}", f"--microbatch {mb}"]
if SMOKE_TEST:
    extras = [
        f"--data_dir {DATA_DIR}",
        f"--validation_dir {DATA_DIR}",
        "--max_steps 5",
        "--save_interval 5",
        "--validation_interval 5",
        "--plot_interval 1000000",
        "--batch_size 2",
        "--microbatch 2",
        "--log_interval 1",
    ]

resume_flag = f"--resume_checkpoint {resume_ckpt}" if resume_ckpt else ""
cmd = f"""set -euo pipefail
source {FLAG_SCRIPT}
cd {DIFFUSION_ROOT}
python3 scripts/autoencoder_train.py $AE_TRAIN_FLAGS \\
  --log_dir {LOG_DIR} \\
  {resume_flag} {' '.join(extras)}
"""
print(cmd)
assert "DNN-ROI-images" in cmd, "launch cmd must use ICARUS DNN-ROI data"
assert "training-SBND" not in cmd and "filelists" not in cmd, "SBND paths leaked into launch cmd"
WRAPPER = LOG_DIR / "launch.sh"
WRAPPER.write_text(cmd)
WRAPPER.chmod(0o755)
print("wrote", WRAPPER)


LOG_DIR  /scratch/7DayLifetime/munjung/anomaly-detection/training/cae/icarus
DURABLE  /exp/sbnd/data/users/munjung/anomaly-detection/training/cae/icarus
resume: None
set -euo pipefail
source /exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/cae_icarus.sh
cd /exp/sbnd/app/users/munjung/anomaly-detection/train/diffusion-anomaly
python3 scripts/autoencoder_train.py $AE_TRAIN_FLAGS \
  --log_dir /scratch/7DayLifetime/munjung/anomaly-detection/training/cae/icarus \
   --max_steps 612000

wrote /scratch/7DayLifetime/munjung/anomaly-detection/training/cae/icarus/launch.sh


In [ ]:
log_file = LOG_DIR / "train.log"
run_env = os.environ.copy()
if SMOKE_TEST:
    run_env["DIFFUSION_TRAINING_TEST"] = "1"

print("MODE", MODE, "DRY_RUN", DRY_RUN, "SMOKE_TEST", SMOKE_TEST, "MAX_STEPS", MAX_STEPS)
if DRY_RUN:
    print("(dry) not launching — set MODE='full' or MODE='smoke' and re-run from controls.")
else:
    stop = APP_ROOT / "STOP_TRAINING"
    if stop.is_file():
        stop.unlink()
        print("removed", stop)
    print("launching", WRAPPER)
    print("This cell blocks until training finishes (or you Interrupt the kernel).")
    with open(log_file, "a") as lf:
        lf.write(f"\n# launch {time.asctime()} AE={AE_TYPE} mode={MODE}\n")
    ret = subprocess.run(["bash", str(WRAPPER)], cwd=str(DIFFUSION_ROOT), env=run_env)
    print("exit", ret.returncode)
    for pattern in (
        "ema_*.pt", "emabrats2update_*.pt", "brats2update*.pt",
        "model*.pt", "progress.csv", "log.txt",
    ):
        for src in LOG_DIR.glob(pattern):
            dest = DURABLE / src.name
            if (not dest.exists()) or src.stat().st_mtime > dest.stat().st_mtime:
                dest.write_bytes(src.read_bytes())
                print("synced", dest)


MODE full DRY_RUN False SMOKE_TEST False MAX_STEPS 612000
launching /scratch/7DayLifetime/munjung/anomaly-detection/training/cae/icarus/launch.sh
This cell blocks until training finishes (or you Interrupt the kernel).
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
port2 58595
setup_dist: backend=gloo CUDA_VISIBLE_DEVICES=0 cuda_available=True device_count=1
setup_dist: device0= NVIDIA A100 80GB PCIe MIG 2g.20gb
Logging to /scratch/7DayLifetime/munjung/anomaly-detection/training/cae/icarus
creating autoencoder (cae)...
creating data loader...
training...


0it [00:00, ?it/s]

-------------------------
| grad_norm  | 4.48     |
| l2         | 0.0891   |
| l2_q0      | 0.0891   |
| loss       | 0.0891   |
| loss_q0    | 0.0891   |
| param_norm | 71.3     |
| samples    | 16       |
| step       | 0        |
-------------------------


50it [01:59,  2.02s/it]

--------------------------
| grad_norm   | 0.501    |
| l2          | 0.0188   |
| l2_q0       | 0.017    |
| loss        | 0.0188   |
| loss_q0     | 0.017    |
| param_norm  | 71.3     |
| samples     | 816      |
| step        | 50       |
| val-l2_q0   | 0.0276   |
| val-loss_q0 | 0.0276   |
--------------------------


100it [03:55,  2.01s/it]

--------------------------
| grad_norm   | 0.101    |
| l2          | 0.0114   |
| l2_q0       | 0.0129   |
| loss        | 0.0114   |
| loss_q0     | 0.0129   |
| param_norm  | 71.3     |
| samples     | 1.62e+03 |
| step        | 100      |
| val-l2_q0   | 0.00364  |
| val-loss_q0 | 0.00364  |
--------------------------


150it [05:52,  2.06s/it]

--------------------------
| grad_norm   | 0.113    |
| l2          | 0.0154   |
| l2_q0       | 0.0165   |
| loss        | 0.0154   |
| loss_q0     | 0.0165   |
| param_norm  | 71.3     |
| samples     | 2.42e+03 |
| step        | 150      |
| val-l2_q0   | 0.0101   |
| val-loss_q0 | 0.0101   |
--------------------------


180it [07:04,  2.16s/it]193it [07:34,  2.25s/it]

## Monitor

```bash
tail -f /scratch/7DayLifetime/munjung/anomaly-detection/training/cae/icarus/train.log
```

After a good run, prefer EMA under `DATA_ROOT/training/cae/icarus/` for
`inference/07_BaselineVAE_CAE_ICARUS.ipynb`.
